# Clase 4 — Unidad 4: Preprocesamiento de Datos Biomédicos

**Curso:** Introducción al Análisis de Datos — 2026, Segundo Semestre
**Material de referencia:** *Python Essentials for Biomedical Data Analysis* — Capítulo 4, "Biomedical Data Preprocessing"

## Objetivos de aprendizaje
- Reconocer los **problemas comunes** de los datos biomédicos (faltantes, outliers, inconsistencias) y evaluar su **calidad**.
- Aplicar técnicas para **manejar datos faltantes** (eliminación e imputación).
- Realizar **transformaciones**: normalización/estandarización, codificación de variables categóricas, ajuste de distribución, binning y reducción de dimensionalidad.
- **Detectar y manejar outliers** con métodos estadísticos.
- Comprender la **integración de datos** y las consideraciones de **privacidad y seguridad**.

## Contenidos de la clase
1. Problemas comunes de los datos biomédicos
2. Calidad de datos
3. Manejo de datos faltantes
4. Transformación de datos
5. Manejo de outliers (valores atípicos)
6. Integración de datos
7. Privacidad y seguridad
8. Ejercicios de práctica

> Esta clase corresponde a la **Unidad 4** del libro guía. Usaremos un **conjunto de datos sintético de pacientes** (con valores faltantes y atípicos inyectados a propósito) que reutilizaremos en toda la clase, para que cada técnica se pueda ejecutar en vivo.

## Preparación del entorno

Instalamos las librerías que usaremos: **pandas** y **numpy** (manejo de datos), **scikit-learn** (preprocesamiento), **scipy** (estadística) y **matplotlib** (gráficos).

In [ ]:
%pip install pandas numpy scikit-learn scipy matplotlib

Creamos el conjunto de datos base `base`: 200 pacientes con variables clínicas. Inyectamos **outliers** (valores atípicos) en `glucosa` y `colesterol`, y **valores faltantes** en las columnas numéricas. En cada sección trabajaremos sobre una **copia** de `base` para que las secciones sean independientes.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 200

base = pd.DataFrame({
    "paciente_id": [f"P{i:03d}" for i in range(1, n + 1)],
    "edad": np.random.randint(18, 90, n),
    "sexo": np.random.choice(["F", "M"], n),
    "glucosa": np.random.normal(100, 20, n).round(1),
    "colesterol": np.random.normal(190, 30, n).round(1),
    "imc": np.random.normal(26, 4, n).round(1),
    "diagnostico": np.random.choice(["Sano", "Prediabetes", "Diabetes"], n, p=[0.6, 0.25, 0.15]),
    "nivel_educacion": np.random.choice(["Básica", "Media", "Universitaria", "Posgrado"], n),
})

# Inyectar outliers en posiciones fijas (valores clínicamente imposibles)
base.loc[[10, 20, 30, 40, 50], "glucosa"] = [300, 350, 20, 400, 15]
base.loc[[11, 21, 31, 41], "colesterol"] = [450, 500, 60, 480]

# Inyectar valores faltantes (en filas 60+ para no borrar los outliers)
rng = np.random.default_rng(0)
for col in ["glucosa", "colesterol", "imc"]:
    idx = rng.choice(range(60, n), size=15, replace=False)
    base.loc[idx, col] = np.nan

print("Dimensiones:", base.shape)
base.head()

## 1. Problemas comunes de los datos biomédicos

Antes de analizar, los datos casi siempre requieren **limpieza**. Los problemas más frecuentes son:

- **Complejidad y heterogeneidad:** mezcla de tipos (numéricos, categóricos, texto, imágenes) y de escalas.
- **Datos faltantes o incompletos:** campos vacíos por errores de registro, pérdidas de seguimiento, etc.
- **Outliers y ruido:** valores extremos por errores de medición o variabilidad biológica real.
- **Inconsistencias y errores:** unidades distintas, formatos mezclados, duplicados, categorías mal escritas.

El **preprocesamiento** transforma datos crudos en datos listos para el análisis, mejorando su calidad y la validez de las conclusiones.

## 2. Calidad de datos

La **calidad de los datos** en el contexto biomédico se evalúa según varias dimensiones:

| Dimensión | Pregunta clave |
|---|---|
| **Exactitud** | ¿Los valores reflejan la realidad? |
| **Completitud** | ¿Faltan datos? |
| **Consistencia** | ¿Los datos se contradicen entre fuentes o columnas? |
| **Validez** | ¿Los valores están dentro de rangos plausibles? (ej. glucosa > 0) |
| **Unicidad** | ¿Hay registros duplicados? |
| **Oportunidad** | ¿Los datos están actualizados? |

Una revisión rápida con Pandas nos da una primera impresión de la calidad.

In [ ]:
datos = base.copy()

# Resumen general: tipos de datos y no-nulos por columna
datos.info()

print("\nEstadísticas descriptivas (numéricas):")
print(datos.describe().round(1))

## 3. Manejo de datos faltantes

### 3.1 Identificar los datos faltantes

El primer paso es **detectar** cuántos y dónde están los valores faltantes (`NaN`).

In [ ]:
datos = base.copy()

# Conteo de faltantes por columna
print("Valores faltantes por columna:")
print(datos.isnull().sum())

# Porcentaje de faltantes
print("\nPorcentaje de faltantes (%):")
print((datos.isnull().mean() * 100).round(1))

In [ ]:
import matplotlib.pyplot as plt

# Mapa de calor de valores faltantes (amarillo = faltante)
plt.figure(figsize=(8, 4))
plt.imshow(datos.isnull(), aspect="auto", cmap="viridis", interpolation="nearest")
plt.xticks(range(len(datos.columns)), datos.columns, rotation=45, ha="right")
plt.ylabel("Filas (pacientes)")
plt.title("Mapa de valores faltantes")
plt.tight_layout()
plt.show()

### 3.2 Tipos de datos faltantes

Entender **por qué** faltan los datos guía el método a usar:

| Tipo | Significado | Ejemplo |
|---|---|---|
| **MCAR** (*Missing Completely At Random*) | La falta no depende de ningún dato. | Pérdida aleatoria por fallo técnico. |
| **MAR** (*Missing At Random*) | La falta depende de datos **observados**. | Cierto examen se omite más en un grupo de edad. |
| **MNAR** (*Missing Not At Random*) | La falta depende del **valor faltante** mismo. | Pacientes con ingresos altos no los reportan. |

MCAR es el más simple de manejar; MNAR es el más complejo y puede sesgar los resultados.

### 3.3 Eliminación de registros incompletos

La forma más directa: **eliminar** filas o columnas con faltantes. Es simple, pero reduce el tamaño de la muestra y puede introducir sesgo si los datos no son MCAR.

In [ ]:
datos = base.copy()
print("Filas originales:", len(datos))

# Eliminación por lista (listwise): quita cualquier fila con al menos un faltante
sin_faltantes = datos.dropna()
print("Filas tras dropna():", len(sin_faltantes))

# También se pueden eliminar solo filas con faltantes en columnas específicas
solo_glucosa = datos.dropna(subset=["glucosa"])
print("Filas con glucosa presente:", len(solo_glucosa))

In [ ]:
datos = base.copy()

# Eliminación por umbral: quitar columnas con más de 20% de faltantes
umbral = 0.20
columnas_ok = datos.loc[:, datos.isnull().mean() <= umbral]
print("Columnas conservadas:", list(columnas_ok.columns))

### 3.4 Imputación simple

En lugar de eliminar, la **imputación** rellena los faltantes con un valor estimado. `SimpleImputer` de scikit-learn permite usar la **media**, la **mediana** o el valor **más frecuente**.

In [ ]:
from sklearn.impute import SimpleImputer

datos = base.copy()
num_cols = ["glucosa", "colesterol", "imc"]

# Imputación por la MEDIA
imp_media = SimpleImputer(strategy="mean")
datos_media = datos.copy()
datos_media[num_cols] = imp_media.fit_transform(datos[num_cols])

print("Faltantes antes:", int(datos[num_cols].isnull().sum().sum()))
print("Faltantes después de imputar:", int(datos_media[num_cols].isnull().sum().sum()))
print(datos_media[num_cols].head())

In [ ]:
# Imputación por la MEDIANA: más robusta ante outliers que la media
imp_mediana = SimpleImputer(strategy="median")
datos_mediana = datos.copy()
datos_mediana[num_cols] = imp_mediana.fit_transform(datos[num_cols])

print("Glucosa imputada -> media:", round(datos_media["glucosa"].mean(), 2),
      "| mediana:", round(datos_mediana["glucosa"].mean(), 2))
print("\nNota: para columnas CATEGÓRICAS se usa strategy='most_frequent' (la moda).")

In [ ]:
# LOCF (Last Observation Carried Forward): propaga el último valor válido hacia adelante.
# Útil en datos longitudinales (mismo paciente medido en el tiempo).
datos = base.copy()
datos_locf = datos.copy()
datos_locf[num_cols] = datos_locf[num_cols].ffill()
print(datos_locf[num_cols].head(8))

### 3.5 Imputación avanzada (basada en modelos)

Métodos que estiman los faltantes a partir de las **relaciones entre variables**:
- **KNN**: usa los `k` vecinos más parecidos.
- **Imputación iterativa** (tipo *MICE*): modela cada variable en función de las demás.

In [ ]:
from sklearn.impute import KNNImputer

datos = base.copy()
imp_knn = KNNImputer(n_neighbors=5)
datos_knn = datos.copy()
datos_knn[num_cols] = imp_knn.fit_transform(datos[num_cols])

print("Faltantes tras imputación KNN:", int(datos_knn[num_cols].isnull().sum().sum()))
print(datos_knn[num_cols].head())

In [ ]:
# Imputación iterativa (tipo MICE): requiere habilitarla explícitamente
from sklearn.experimental import enable_iterative_imputer  # noqa: habilita IterativeImputer
from sklearn.impute import IterativeImputer

datos = base.copy()
imp_iter = IterativeImputer(random_state=42)
datos_iter = datos.copy()
datos_iter[num_cols] = imp_iter.fit_transform(datos[num_cols])

print("Faltantes tras imputación iterativa:", int(datos_iter[num_cols].isnull().sum().sum()))
print(datos_iter[num_cols].head())

## 4. Transformación de datos

Transformar los datos los hace más adecuados para el análisis y mejora el desempeño de los modelos. Veremos: normalización/estandarización, codificación de categóricas, ajuste de distribución, binning y reducción de dimensionalidad.

Para las siguientes secciones usaremos un conjunto **sin faltantes** (imputado por la mediana).

In [ ]:
from sklearn.impute import SimpleImputer

num_cols = ["glucosa", "colesterol", "imc"]
completo = base.copy()
completo[num_cols] = SimpleImputer(strategy="median").fit_transform(completo[num_cols])
print("Faltantes restantes:", int(completo[num_cols].isnull().sum().sum()))

### 4.1 Normalización y estandarización

Las variables biomédicas suelen tener **escalas muy distintas** (ej. edad 18-90 vs. colesterol 60-500). Escalarlas evita que una domine el análisis.

- **Min-Max scaling** (normalización): lleva los valores al rango [0, 1].
- **Z-score / StandardScaler** (estandarización): media 0 y desviación 1.
- **RobustScaler**: usa mediana e IQR, menos sensible a outliers.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Min-Max: rango [0, 1]
minmax = MinMaxScaler()
escalado = pd.DataFrame(minmax.fit_transform(completo[num_cols]), columns=num_cols)
print("Min-Max (observe min=0, max=1):")
print(escalado.describe().loc[["min", "max"]].round(2))

In [ ]:
from sklearn.preprocessing import StandardScaler

# Z-score: media ≈ 0, desviación ≈ 1
estandar = StandardScaler()
estandarizado = pd.DataFrame(estandar.fit_transform(completo[num_cols]), columns=num_cols)
print("Z-score (media≈0, std≈1):")
print(estandarizado.describe().loc[["mean", "std"]].round(2))

In [ ]:
from sklearn.preprocessing import RobustScaler

# Robust: usa mediana e IQR (recomendado si hay outliers)
robusto = RobustScaler()
robusto_df = pd.DataFrame(robusto.fit_transform(completo[num_cols]), columns=num_cols)
print(robusto_df.head())

### 4.2 Codificación de variables categóricas

Los modelos requieren números, no texto. Cada método sirve para un tipo de categoría:
- **One-hot**: para variables **nominales** (sin orden), crea una columna binaria por categoría.
- **Label encoding**: asigna un entero a cada categoría.
- **Ordinal encoding**: para variables con **orden** (ej. nivel educativo).

In [ ]:
import pandas as pd
datos = base.copy()

# One-hot encoding con pandas get_dummies (variables nominales)
onehot = pd.get_dummies(datos[["sexo", "diagnostico"]])
print(onehot.head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encoding: un entero por categoría
le = LabelEncoder()
datos["sexo_cod"] = le.fit_transform(datos["sexo"])
print("Clases:", list(le.classes_))
print(datos[["sexo", "sexo_cod"]].head())

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Ordinal encoding: respeta el orden de las categorías
orden = [["Básica", "Media", "Universitaria", "Posgrado"]]
oe = OrdinalEncoder(categories=orden)
datos["educacion_cod"] = oe.fit_transform(datos[["nivel_educacion"]])
print(datos[["nivel_educacion", "educacion_cod"]].drop_duplicates().sort_values("educacion_cod"))

### 4.3 Manejo de la distribución

Muchas técnicas asumen distribución **normal**. Cuando los datos están **sesgados**, transformaciones como el **logaritmo** o **Box-Cox** los acercan a la normalidad.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
# Datos sesgados a la derecha (ej. niveles de expresión génica)
expresion = np.random.exponential(scale=1000, size=500)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(expresion, bins=30, color="#4C78A8")
ax[0].set_title("Original (sesgada a la derecha)")
ax[1].hist(np.log2(expresion + 1), bins=30, color="#59A14F")
ax[1].set_title("Tras transformación log2")
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats
import numpy as np

np.random.seed(42)
# Presión sistólica simulada (sesgada). Box-Cox busca el lambda óptimo.
presion = np.random.gamma(shape=2.0, scale=15.0, size=500) + 100
transformada, lam = stats.boxcox(presion)
print(f"Lambda óptimo de Box-Cox: {lam:.4f}")
print("Media original:", round(presion.mean(), 2), "| Media transformada:", round(transformada.mean(), 2))

### 4.4 Binning (discretización)

Convertir una variable **continua** en **categorías** (intervalos o "bins"). Útil para simplificar el análisis o crear grupos clínicos.
- `pd.cut`: intervalos de **ancho fijo** (o límites definidos por nosotros).
- `pd.qcut`: intervalos de **igual frecuencia** (cuantiles).

In [ ]:
import pandas as pd
datos = base.copy()

# Binning por rangos definidos sobre la edad
datos["grupo_edad"] = pd.cut(
    datos["edad"],
    bins=[18, 30, 45, 60, 90],
    labels=["18-30", "31-45", "46-60", "61+"],
    include_lowest=True,
)
print(datos["grupo_edad"].value_counts().sort_index())

In [ ]:
# Binning por cuartiles (igual frecuencia) sobre la glucosa
completo_glu = base.copy()
completo_glu["glucosa"] = completo_glu["glucosa"].fillna(completo_glu["glucosa"].median())
completo_glu["glucosa_q"] = pd.qcut(completo_glu["glucosa"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print(completo_glu["glucosa_q"].value_counts().sort_index())

### 4.5 Reducción de dimensionalidad (PCA)

Cuando hay **muchas variables** (ej. miles de genes), el **Análisis de Componentes Principales (PCA)** las resume en pocas componentes que conservan la mayor parte de la varianza. Es clave estandarizar antes de aplicarlo.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Datos sintéticos de expresión génica: 100 muestras, 50 genes
np.random.seed(42)
X = np.random.normal(size=(100, 50))
X_esc = StandardScaler().fit_transform(X)

pca = PCA(n_components=5)
componentes = pca.fit_transform(X_esc)

print("Varianza explicada por componente:", pca.explained_variance_ratio_.round(3))
print("Varianza acumulada:", np.cumsum(pca.explained_variance_ratio_).round(3))

In [ ]:
import matplotlib.pyplot as plt

# Gráfico de varianza explicada (scree plot)
plt.figure(figsize=(6, 4))
comp = range(1, len(pca.explained_variance_ratio_) + 1)
plt.bar(comp, pca.explained_variance_ratio_, color="#4C78A8")
plt.xlabel("Componente principal")
plt.ylabel("Proporción de varianza explicada")
plt.title("Scree plot (PCA)")
plt.tight_layout()
plt.show()

## 5. Manejo de outliers (valores atípicos)

Los **outliers** son puntos que se alejan mucho del resto. Pueden deberse a errores o a variabilidad biológica real. Detectarlos es importante porque afectan medias, desviaciones y modelos.

Métodos estadísticos frecuentes:
- **Z-score**: puntos con |Z| > 3 (asume distribución normal).
- **IQR**: puntos fuera de [Q1 − 1.5·IQR, Q3 + 1.5·IQR] (no asume normalidad).

In [ ]:
import numpy as np
from scipy import stats

def deteccion_zscore(data, umbral=3):
    """Devuelve los valores cuyo |Z-score| supera el umbral."""
    z = np.abs(stats.zscore(data))
    return data[z > umbral]

datos = base.copy()
glu = datos["glucosa"].dropna().to_numpy()

outliers_z = deteccion_zscore(glu)
print("Outliers en glucosa por Z-score:", np.sort(outliers_z))

In [ ]:
def deteccion_iqr(data):
    """Detecta outliers con el método del rango intercuartílico (IQR)."""
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = data[(data < lim_inf) | (data > lim_sup)]
    return outliers, lim_inf, lim_sup

outliers_iqr, inf, sup = deteccion_iqr(glu)
print(f"Límites IQR: [{inf:.1f}, {sup:.1f}]")
print("Outliers en glucosa por IQR:", np.sort(outliers_iqr))

In [ ]:
import matplotlib.pyplot as plt

# Boxplot: los puntos fuera de los "bigotes" son outliers según el criterio IQR
plt.figure(figsize=(7, 3))
plt.boxplot(glu, vert=False)
plt.title("Boxplot de glucosa (los puntos aislados son outliers)")
plt.xlabel("Glucosa (mg/dL)")
plt.tight_layout()
plt.show()

**Estrategias para manejar outliers** una vez detectados:
- **Eliminarlos** (si son errores claros).
- **Recortarlos / winsorizar** (reemplazar por los límites).
- **Transformar** la variable (ej. log).
- **Conservarlos** si representan casos clínicos reales importantes.

Otros métodos más avanzados (no cubiertos aquí en detalle): basados en distancia (KNN), en modelos (*Isolation Forest*), en agrupamiento (*DBSCAN*) y de proximidad (LOF).

In [ ]:
# Ejemplo de recorte (winsorización) usando los límites del IQR
datos = base.copy()
datos_recortado = datos.copy()
datos_recortado["glucosa"] = datos_recortado["glucosa"].clip(lower=inf, upper=sup)

print("Glucosa máx antes:", datos["glucosa"].max(), "-> después:", datos_recortado["glucosa"].max())
print("Glucosa mín antes:", datos["glucosa"].min(), "-> después:", datos_recortado["glucosa"].min())

## 6. Integración de datos

En la práctica, los datos provienen de **múltiples fuentes** (demografía, laboratorio, imágenes). Integrarlos requiere una **clave común** y resolver inconsistencias y duplicados.

In [ ]:
import pandas as pd

# Dos tablas que comparten la clave 'paciente_id'
demograficos = base[["paciente_id", "edad", "sexo"]].head(5)
laboratorio = base[["paciente_id", "glucosa", "colesterol"]].head(5)

# Integración (merge/join) por la clave común
integrado = pd.merge(demograficos, laboratorio, on="paciente_id")
print(integrado)

In [ ]:
# Detección y eliminación de duplicados
con_duplicado = pd.concat([integrado, integrado.iloc[[0]]], ignore_index=True)
print("Con duplicado:", len(con_duplicado), "filas")
print("¿Filas duplicadas?", con_duplicado.duplicated().sum())
print("Tras drop_duplicates():", len(con_duplicado.drop_duplicates()), "filas")

## 7. Privacidad y seguridad

Los datos biomédicos contienen información **sensible**. Antes de compartirlos hay que protegerlos mediante técnicas de **anonimización**:
- **Seudonimización:** reemplazar identificadores directos por códigos.
- **Generalización:** reducir el detalle (ej. edad exacta → rango de edad).
- **Supresión:** eliminar columnas altamente identificatorias.

In [ ]:
import pandas as pd
datos = base.copy()

# Seudonimización: reemplazar el identificador por un código anónimo
datos["paciente_id"] = ["ANON_" + str(i) for i in range(len(datos))]

# Generalización: agrupar la edad en rangos (reduce la identificabilidad)
datos["edad_rango"] = pd.cut(datos["edad"], bins=[0, 30, 45, 60, 120],
                             labels=["<=30", "31-45", "46-60", "60+"])

print(datos[["paciente_id", "edad_rango", "sexo", "diagnostico"]].head())

## 8. Ejercicios de práctica

Usa el conjunto `base` (o crea el tuyo) para resolver:

1. **Identificar faltantes:** muestra el número y el porcentaje de valores faltantes de cada columna de `base`.
2. **Eliminar vs. imputar:** compara cuántas filas quedan tras `dropna()` frente a imputar `colesterol` con la mediana.
3. **Imputación:** imputa las columnas numéricas con `KNNImputer` y compara la media de `imc` antes y después.
4. **Estandarización:** aplica `StandardScaler` a `edad`, `glucosa` y `colesterol`, y verifica que la media quede ≈ 0.
5. **Codificación:** aplica one-hot encoding a `diagnostico` y ordinal encoding a `nivel_educacion`.
6. **Binning:** crea una variable categórica de `imc` con las categorías OMS: `<18.5`, `18.5-25`, `25-30`, `>=30`.
7. **Outliers:** detecta los outliers de `colesterol` con el método IQR e imprímelos.
8. **PCA:** genera 30 variables aleatorias, estandarízalas y aplica PCA; ¿cuánta varianza explican las 2 primeras componentes?

### Preguntas para reflexionar
1. ¿Qué diferencia hay entre MCAR, MAR y MNAR, y por qué importa para elegir el método de imputación?
2. ¿Cuándo conviene la mediana en lugar de la media para imputar?
3. ¿Por qué se estandarizan las variables antes de aplicar PCA o KNN?
4. ¿Qué método de codificación usarías para una variable ordinal y por qué?
5. ¿Siempre se deben eliminar los outliers? Da un ejemplo clínico donde NO.

In [ ]:
# Espacio para resolver los ejercicios
